## Vector Search Index Sync

This notebook manually triggers a sync for a Delta-based vector search index using the Databricks Vector Search client.

It retrieves the specified index, initiates a sync operation, and optionally waits until the sync completes (i.e., the index reaches an `ONLINE_NO_PENDING_UPDATE` state).

Use this notebook after updating your index input table to ensure that all new or changed records are embedded and reflected in the index before querying.


In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
# Environment Variables
CATALOG = "mlb_tech_summit"
SCHEMA = "rag_demo"
TABLE = "silver_prospects"

# Vector Store Environment Variables
INDEX_NAME = "scouting_reports_index"
EMBEDDING_MODEL = "databricks-gte-large-en"  # your model serving endpoint
VECTOR_SEARCH_ENDPOINT = "abooth_vs"
PRIMARY_KEY = "primary_key"
SOURCE_COLUMN = "combined_text"

In [0]:
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

In [0]:
import time

# Define index path (same as in UI)
FULL_INDEX_NAME = f"{CATALOG}.{SCHEMA}.{INDEX_NAME}"

# Retrieve the index object
index = vsc.get_index(endpoint_name=VECTOR_SEARCH_ENDPOINT, index_name=FULL_INDEX_NAME)

# Kick off Sync
index.sync()

# Optional: Wait until sync is complete (no pending update)
MAX_WAIT_SECONDS = 300
POLL_INTERVAL = 10
start = time.time()

while True:
    status = index.describe()
    state = status["status"]["detailed_state"]
    print(f"Current status: {state}")

    if state == "ONLINE_NO_PENDING_UPDATE":
        print("✅ Index sync completed.")
        break

    if time.time() - start > MAX_WAIT_SECONDS:
        raise TimeoutError("❌ Sync did not complete in time.")

    time.sleep(POLL_INTERVAL)

print("✅ Index is ready for queries.")

In [0]:
status_info = index.describe()
print(f"Index status: {status_info['status']}")